# `Bloom-Filter-Hash`

Example notebook on how to train and break hashes using the `bloom-filter-hash` python package.

Overall this provides an alternative way to do some pre-work too try and reduce the brute force needed to break hashes.

## Train

Let us create a set of bloom filters to break password md5 hashes of only lower case characters of length 4. 

In [ ]:
from bloom_filter_hash import train
import string

train(
    charset=set(list(string.ascii_lowercase)),
    password_length=4,
    hash_alg="md5",
)

## Break Hash

Now that we have a set of trained filters, let us use this to break some hashes.

In [ ]:
# Generate some example hashes to break
import hashlib

# A hash we can break
hash1 = hashlib.md5("test".encode()).hexdigest()
# A hash we cannot break due to it containing un upper case char
hash2 = hashlib.md5("Test".encode()).hexdigest()

In [ ]:
from bloom_filter_hash import break_hash

pwd1 = break_hash(hash=hash1, hash_alg="md5")

pwd2 = break_hash(hash=hash2, hash_alg="md5")

pwd1, pwd2
# >>> ('test', None)
# As we can see we have found the hash for the first

## Get Charset

If you are only interested in the bloom filter hits you can use the `get_charset` function.

In [ ]:
from bloom_filter_hash import get_charset

charsets = get_charset(hash1, hash_alg="md5")
charsets

# Output:
# {
#     'pretrained_filters/md5/4': {
#         'password_length': 4,
#         'charset_hit': {'e', 's', 't'}
#     }
# }

## HashCat Command

Rather than using python to break the hash generate the associated HashCat command to utilize the performance of HashCat.

In [ ]:
import hashlib
from bloom_filter_hash import hashcat

# Generate hash to break
hash = hashlib.md5("test".encode()).hexdigest()

command = hashcat(hash, hash_alg="md5")
command

In [ ]:
# Here is the outputted command ready to be run
!hashcat -m 0 -a 3 098f6bcd4621d373cade4e832627b4f6 --custom-charset1 tse ?1?1?1?1

## Pre-trained filters

Here is how you can download and use some of my pre-trained filters. You can find them on HuggingFace [here](https://huggingface.co/sw241395/bloom-filter-hash)

In [ ]:
# Download from Hugging Face
! git clone https://huggingface.co/sw241395/bloom-filter-hash
# Should see a bloom-filter-hash folder

In [ ]:
# Unzip the pre-trained filters for passwords of 4 chars
! tar -xvzf ./bloom-filter-hash//sha256/4.tar.gz
# Should see a folder called 4 with 0 - 92 .bin files, these are our pre-trained filters

In [ ]:
# Break hash using the downloaded filters

import hashlib
from bloom_filter_hash import break_hash

# Generate hash to break
hash = hashlib.sha256("test".encode()).hexdigest()

# Break hash using downloaded pre-trained filters
pwd = break_hash(hash=hash, hash_alg="sha256", path_to_filters="./4")
pwd

# Positional Filters

In addition the the filters above where returned is a list of chars that are potentially in the password, you can create a more advanced set of filters to find out what chars are in what position in the password.

Fpr example if you had a `md5` hash of the password "test", you can train a set of filters to indicate that:

0. "t" in position 0
1. "e" in position 1
2. "s" in position 2
3. "t" in position 3

In [ ]:
from bloom_filter_hash import train_positions
import string

# Train a set of positional filters on lower case chars of length 4
train_positions(
    charset=set(list(string.ascii_lowercase)),
    password_length=4,
    hash_alg="md5",
)

In [ ]:
import hashlib
from bloom_filter_hash import break_hash

# A hash we can break
hash1 = hashlib.md5("test".encode()).hexdigest()

# Using the same break_hash method as before just point it to where the position filters are stored to use the positional filters
pwd1 = break_hash(
    hash=hash1, hash_alg="md5", path_to_filters="./pretrained_position_filters"
)
pwd1